In [ ]:
import earthkit.data
import earthkit.plots
from earthkit.geo import regrid, cartography

# Polytope Examples for Area
## Operational Forecast Data

<div style="border:2px solid #f39c12; background-color:#fcf8e3; padding:10px; border-radius:5px;">
  <strong>⚠️ Warning:</strong> Operational FDB only retains data for 2 days in the past.
</div>


## 1) Global Request - Area Bounded

### Server side Regridding (reduced_gg -> lat/lon)

In [ ]:
request_1 = {
    "class": "od",
    "date": "0",
    "expver": 1,
    "stream": "oper",
    "time": "00",
    "type": "fc",
    "levtype": "sfc",
    "param": "167",
    "step": 0,
    "grid": "0.1/0.1", # regrid to lat lon because orig in reduced gaussian
    "area": "60/-20/40/5", #+N, -W, -S, +E
}
data_1 = earthkit.data.from_source(
    "polytope", "ecmwf-mars", request_1, address="polytope.ecmwf.int", stream=False)

In [ ]:
data_1 = data_1.to_fieldlist()
data_1.ls() # gridType is now regular in metadata

In [ ]:
data_1.to_target("file", "../data/data_1.grib") # writes the fieldlist back out as a .grib file

In [ ]:
chart = earthkit.plots.Map()
chart.contourf(
    data_1,
    units="celsius"
)
chart.legend()
chart.title()
chart.coastlines()
chart.gridlines()
chart.show()

## 2) Feature Request - Bounding Box

### Polytope Bounding Box Feature [Documentation](https://polytope.readthedocs.io/en/latest/Service/Features/boundingbox/)

### No server side regridding possible

In [ ]:
request_2 = {
"class": "od",
    "date": "-1",
    "expver": 1,
    "stream": "oper",
    "time": "00",
    "type": "fc",
    "levtype": "sfc",
    "param": "167",
    "step": 0,
    "feature": {
        "type": "boundingbox",
        "points" : [[60, -20], [40, 5]]  #top left (lat lon) #bottom right (lat, lon)
    },
}
data_2 = earthkit.data.from_source(
    "polytope", "ecmwf-mars", request_2, address="polytope.ecmwf.int", stream=False)

In [ ]:
type(data_2) # data is returned as CovJsonData

In [ ]:
data_2.to_geojson() #  display as GeoJSON format

In [ ]:
dx_2 = data_2.to_xarray() # convert the data to an xarray object for easier manipulation
dx_2

In [ ]:
chart = earthkit.plots.Map()
chart.contourf(
    dx_2["2t"],
    x='longitude',
    y='latitude',
    units="celsius"
)
chart.legend()
chart.coastlines()
chart.gridlines()
chart.show()

## 3) Global Request - Area Bounded with Earthkit Plots Domain

### Client Side Regridding

In [ ]:
request_3 = {
    "class": "od",
    "date": "-1",
    "expver": 1,
    "stream": "oper",
    "time": "00",
    "type": "fc",
    "levtype": "sfc",
    "param": "167",
    "step": 0,
}
data_3 = earthkit.data.from_source(
    "polytope", "ecmwf-mars", request_3, address="polytope.ecmwf.int", stream=False)

In [ ]:
data_3 = data_3.to_fieldlist()
data_3.ls() # data is in its original reduced gausian grid

In [ ]:
# Client Side Regridding
data_latlon = regrid(
    data_3,
    out_grid={"grid": [0.5, 0.5]},  # example: interpolate to 0.5°x0.5° lat-lon grid
    interpolation="linear"
)

In [ ]:
data_latlon.ls() # Now in regular grid

In [ ]:
custom_domain = earthkit.plots.geography.domains.Domain.from_bbox(
    bbox=[-20, 5, 40, 60], #x_min, x_max, y_min, y_max
)

In [ ]:
chart = earthkit.plots.Map(domain=custom_domain)
chart.contourf(
    data_latlon,
    units="celsius"
)
chart.legend()
chart.title()
chart.coastlines()
chart.gridlines()
chart.show()

## 4) Feature Extraction - Polygon Area

### Polytope Polygon Feature Extraction [Documentation](https://polytope.readthedocs.io/en/latest/Service/Features/polygon/)

In [ ]:
countries = ["Germany"] # List of countries possiblie

shape = cartography.country_polygons(countries, resolution=50e6)


polytope_address = "polytope.ecmwf.int"

request_4 = {
"class": "od",
    "date": "-1",
    "expver": 1,
    "stream": "oper",
    "time": "00",
    "type": "fc",
    "levtype": "sfc",
    "param": "167",
    "step": 0,
    "feature": {
        "type": "polygon",
        "shape": shape
    },
}
data_4 = earthkit.data.from_source(
    "polytope", "ecmwf-mars", request_4, address="polytope.ecmwf.int", stream=False)

In [ ]:
dx_4 = data_4.to_xarray()

In [ ]:
chart = earthkit.plots.Map(domain=["Germany"])
chart.contourf(
    dx_4["2t"],
    x='longitude',
    y='latitude',
    units="celsius"
)

chart.coastlines()
chart.borders()
chart.gridlines()

# chart.title()

chart.legend()

chart.show()